# 05 · Fejlanalyse

## Opsætning

In [ ]:
# Find training/ (the folder with pyproject.toml), wherever the notebook runs from.
import sys
from pathlib import Path

TRAINING = next(c for p in [Path.cwd(), *Path.cwd().parents] for c in (p, p / "training")
                if (c / "pyproject.toml").is_file() and (c / "wt_train").is_dir())
if str(TRAINING) not in sys.path:
    sys.path.insert(0, str(TRAINING))
print(TRAINING)

In [ ]:
import json
import re
import textwrap
from collections import Counter, defaultdict
from html import escape

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Audio, display

from wt_train import audio, metrics, paths
from wt_train.spelling import load_spelling


def da(x, digits=0):
    """A number written the Danish way: 20.464 and 6,32."""
    return f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def table(header, rows, right=(), empty="ingen rækker"):
    """A small HTML table; columns in `right` are right-aligned."""
    def cell(tag, i, v):
        align = "right" if i in right else "left"
        return f'<{tag} style="text-align:{align}">{escape(str(v))}</{tag}>'
    head = "".join(cell("th", i, h) for i, h in enumerate(header))
    body = "".join("<tr>" + "".join(cell("td", i, v) for i, v in enumerate(r)) + "</tr>" for r in rows)
    if not rows:
        body = f'<tr><td colspan="{len(header)}" style="text-align:left"><i>{escape(empty)}</i></td></tr>'
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

## Vælg en evaluering

In [ ]:
# Every `wt-train eval` leaves a folder in <work>/runs/eval/; this notebook only reads them.
runs = sorted(p for p in paths.eval_runs_dir().glob("*") if (p / "summary.json").is_file())
assert runs, f"ingen evalueringer i {paths.eval_runs_dir()}: kør wt-train eval først"
listing = []
for run in runs:
    s = json.loads((run / "summary.json").read_text(encoding="utf-8"))
    for name, result in s["results"].items():
        listing.append((run.name, s["backend"], name, result["n"], da(result["wer"] * 100, 2)))
table(["kørsel", "backend", "datasæt", "klip", "ordfejl (WER) %"], listing, right={3, 4})

In [ ]:
RUN = None      # a name from the table above; None = the latest
DATASET = None  # one of that run's datasets; None = the one it was run on

run = paths.eval_runs_dir() / RUN if RUN else runs[-1]
summary = json.loads((run / "summary.json").read_text(encoding="utf-8"))
dataset = DATASET or next(iter(summary["results"]))


def slug(text):
    """How wt-train eval names a dataset's row file."""
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:60] or "model"


rows_file = next(f for f in (run / f"{slug(dataset)}.jsonl", run / f"{dataset}.jsonl") if f.is_file())
rows = [json.loads(line) for line in rows_file.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"kørsel:  {run.name}")
print(f"model:   {summary['model']} ({summary['backend']})")
print(f"datasæt: {dataset}: {len(rows)} klip, fra {rows_file.name}")
print(textwrap.fill(f"felter i hver række: {', '.join(rows[0])}", 100, subsequent_indent=" " * 21))

## Overblik

In [ ]:
for r in rows:
    r["errors"] = r["sub"] + r["del"] + r["ins"]
    r["wer"] = r["errors"] / r["ref_words"] if r["ref_words"] else float(r["errors"] > 0)

s = metrics.summarize(rows)  # the workbench's own scoring, from the rows
low, high = s["wer_ci95"]
print(f"ordfejl (WER): {da(s['wer'] * 100, 2)} % (95 %-interval {da(low * 100, 2)} til {da(high * 100, 2)} %); "
      f"tegnfejl (CER): {da(s['cer'] * 100, 2)} %")
print(f"samme WER som summary.json: {'ja' if abs(s['wer'] - summary['results'][dataset]['wer']) < 1e-9 else 'nej'}")
print(f"{da(s['reference_words'])} ord i referencerne; fejlene: {da(sum(r['sub'] for r in rows))} ombyttede, "
      f"{da(sum(r['del'] for r in rows))} udeladte og {da(sum(r['ins'] for r in rows))} indsatte ord")
perfect = sum(1 for r in rows if r["errors"] == 0)
print(f"klip uden fejl: {perfect} af {len(rows)} ({da(perfect / len(rows) * 100, 1)} %)")

per_clip = Counter(min(r["errors"], 10) for r in rows)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(11), [per_clip[k] for k in range(11)], color="#4c6ef5")
ax.set_xticks(range(11), [str(k) for k in range(10)] + ["10+"])
ax.set(xlabel="ordfejl i klippet", ylabel="klip", title="Hvor mange fejl har hvert klip")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Ordene stillet op

In [ ]:
def align(ref, hyp):
    """Word alignment with the costs and tie order of models/eval/wt_eval/wer.py (substitution,
    then deletion, then insertion), so its counts are the row's. Returns (op, ref word, hyp word):
    "=" the same, "S" substituted, "D" deleted (missing), "I" inserted (extra)."""
    n, m = len(ref), len(hyp)
    cost = [[0] * (m + 1) for _ in range(n + 1)]
    back = [[""] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        cost[i][0], back[i][0] = i, "D"
    for j in range(1, m + 1):
        cost[0][j], back[0][j] = j, "I"
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                cost[i][j], back[i][j] = cost[i - 1][j - 1], "="
            else:
                cost[i][j], back[i][j] = min((cost[i - 1][j - 1] + 1, "S"), (cost[i - 1][j] + 1, "D"),
                                             (cost[i][j - 1] + 1, "I"), key=lambda t: t[0])
    ops, i, j = [], n, m
    while i or j:  # walk back from the end
        op = back[i][j]
        if op in ("=", "S"):
            ops.append((op, ref[i - 1], hyp[j - 1]))
            i, j = i - 1, j - 1
        elif op == "D":
            ops.append((op, ref[i - 1], ""))
            i -= 1
        else:
            ops.append((op, "", hyp[j - 1]))
            j -= 1
    return ops[::-1]


for r in rows:
    r["ops"] = align(r["ref_norm"].split(), r["hyp_norm"].split())
agree = sum(1 for r in rows if [sum(op == k for op, _, _ in r["ops"]) for k in "SDI"] == [r["sub"], r["del"], r["ins"]])
print(f"justeringen giver rækkens egne tal i {agree} af {len(rows)} klip")

# One short clip with more than one kind of error, word by word.
mixed = [r for r in rows if r["ref_words"] <= 15 and r["sub"] and r["del"] + r["ins"]] or rows
example = max(mixed, key=lambda r: (r["errors"], r["id"]))
print(f"\n{example['id']}: {example['sub']} ombyttede, {example['del']} udeladte, {example['ins']} indsatte")
print(f"     {'referencen':<22} modellen")
for op, a, b in example["ops"]:
    print(f"  {op}  {a:<22} {b}")

## De værste klip

In [ ]:
TOP, MIN_WORDS = 5, 8  # WER on a 3-word clip says little, so the WER list needs a few words
by_errors = sorted(rows, key=lambda r: (-r["errors"], r["id"]))[:TOP]
by_wer = sorted((r for r in rows if r["ref_words"] >= MIN_WORDS), key=lambda r: (-r["wer"], r["id"]))[:TOP]
table(["flest fejl", "ord", "fejl", "WER %"],
      [(r["id"], r["ref_words"], r["errors"], da(r["wer"] * 100, 1)) for r in by_errors], right={1, 2, 3})
table([f"højest WER (mindst {MIN_WORDS} ord)", "ord", "fejl", "WER %"],
      [(r["id"], r["ref_words"], r["errors"], da(r["wer"] * 100, 1)) for r in by_wer], right={1, 2, 3})

In [ ]:
RED, GREEN = "#e03131", "#2f9e44"


def diff_html(ops):
    """Red and struck through: only in the reference. Green and underlined: only in what the model wrote."""
    out = []
    for op, a, b in ops:
        gone = f'<span style="color:{RED};text-decoration:line-through">{escape(a)}</span>'
        extra = f'<span style="color:{GREEN};text-decoration:underline;font-weight:600">{escape(b)}</span>'
        out.append({"=": escape(a), "S": f"{gone} {extra}", "D": gone, "I": extra}[op])
    return " ".join(out)


display(HTML(f'<p>Ord for ord: <span style="color:{RED};text-decoration:line-through">rød og overstreget</span> '
             f'står kun i referencen; <span style="color:{GREEN};text-decoration:underline;font-weight:600">'
             f'grøn og understreget</span> skrev modellen, men det står ikke i referencen.</p>'))
worst = list({r["id"]: r for r in by_errors + by_wer}.values())
for r in worst:
    display(HTML(f"<h4>{escape(r['id'])}: {r['errors']} fejl i {r['ref_words']} ord, WER {da(r['wer'] * 100, 1)} % "
                 f"({r['sub']} ombyttede, {r['del']} udeladte, {r['ins']} indsatte)</h4>"))
    wav = Path(r["wav"])
    if wav.is_file():
        display(Audio(np.clip(audio.read_16k_mono(wav), -1, 1), rate=audio.SAMPLE_RATE, normalize=False))
    else:
        print(f"lyden findes ikke længere: {wav}")
    display(HTML(f"<p><b>reference:</b> {escape(r['reference'])}<br>"
                 f"<b>modellen skrev:</b> {escape(r['hypothesis']) or '<i>(ingenting)</i>'}<br>"
                 f"<b>ord for ord:</b> {diff_html(r['ops'])}</p>"))

## Hyppigste ombytninger

In [ ]:
subs = Counter((a, b) for r in rows for op, a, b in r["ops"] if op == "S")
clips_with = defaultdict(set)
for r in rows:
    for op, a, b in r["ops"]:
        if op == "S":
            clips_with[(a, b)].add(r["id"])
print(f"{da(sum(subs.values()))} ombytninger i alt, {da(len(subs))} forskellige par")
table(["nr", "referencen siger", "modellen skrev", "gange", "i klip"],
      [(n, a, b, k, len(clips_with[(a, b)])) for n, ((a, b), k) in enumerate(subs.most_common(20), 1)],
      right={0, 3, 4})

In [ ]:
top = subs.most_common(15)[::-1]
fig, ax = plt.subplots(figsize=(8, 0.34 * len(top) + 1.3))
ax.barh([f"{a} → {b}" for (a, b), _ in top], [k for _, k in top], color="#e8590c")
ax.set(xlabel="gange", title="Hyppigste ombytninger (referencen → modellen)")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Udeladte og indsatte ord

In [ ]:
deleted = Counter(a for r in rows for op, a, _ in r["ops"] if op == "D").most_common(12)
inserted = Counter(b for r in rows for op, _, b in r["ops"] if op == "I").most_common(12)
table(["nr", "udeladt ord", "gange", "indsat ord", "gange"],
      [(n + 1, *(deleted[n] if n < len(deleted) else ("", "")), *(inserted[n] if n < len(inserted) else ("", "")))
       for n in range(max(len(deleted), len(inserted)))], right={0, 2, 4})

## Gentagelser og hallucinationer

In [ ]:
def loops(words, min_repeats=3, max_n=6):
    """(n-gram, times) wherever the same n words come min_repeats or more times in a row."""
    found, i = [], 0
    while i < len(words):
        best = None
        for n in range(1, max_n + 1):
            gram, k = words[i:i + n], 1
            if len(gram) < n:
                break
            while words[i + k * n:i + (k + 1) * n] == gram:
                k += 1
            if k >= min_repeats and (best is None or n * k > best[0] * best[1]):
                best = (n, k)
        if best:
            found.append((" ".join(words[i:i + best[0]]), best[1]))
            i += best[0] * best[1]
        else:
            i += 1
    return found


found = defaultdict(list)
for r in rows:
    ref, hyp = r["ref_norm"].split(), r["hyp_norm"].split()
    # A loop counts when the reference does not say the same thing as many times itself.
    reps = [(g, k) for g, k in loops(hyp) if f" {' '.join([g] * k)} " not in f" {' '.join(ref)} "]
    if reps:
        found["loop"].append((r, ", ".join(f"'{g}' {k} gange" for g, k in reps)))
    if len(hyp) >= 1.5 * len(ref) and len(hyp) - len(ref) >= 5:
        found["long"].append((r, f"{len(hyp)} ord mod {len(ref)}"))
    tail = 0
    while tail < len(r["ops"]) and r["ops"][-1 - tail][0] == "I":
        tail += 1
    if tail >= 3:
        found["tail"].append((r, " ".join(b for _, _, b in r["ops"][-tail:])))
    if ref and not hyp:
        found["empty"].append((r, "tom tekst"))
    if r.get("error"):
        found["failed"].append((r, r["error"]))

checks = {
    "loop": "Gentagelser: det samme ord eller de samme ord 3 gange i træk",
    "long": "Meget længere end referencen: mindst 1,5 gange så mange ord og 5 flere",
    "tail": "Indsatte ord til sidst: 3 eller flere",
    "empty": "Tomme tekster",
    "failed": "Klip der fejlede",
}
for key, label in checks.items():
    items = found[key]
    display(HTML(f"<p><b>{escape(label)}:</b> {len(items) if items else 'ingen fundet'}</p>"))
    if items:
        table(["klip", "fundet", "modellen skrev"], [(r["id"], what, r["hypothesis"]) for r, what in items[:10]])

## Pr. optageforhold

In [ ]:
# The owner's takes carry their recording condition; FLEURS clips have none.
by_condition = defaultdict(list)
for r in rows:
    if r.get("condition"):
        by_condition[r["condition"]].append(r)


def pct(x):
    return "" if x is None else da(x * 100, 1)


condition_rows = []
for cid, group in sorted(by_condition.items()):
    g = metrics.summarize(group)
    condition_rows.append((cid, g["n"], g["reference_words"], da(g["wer"] * 100, 2), da(g["cer"] * 100, 2),
                           pct(g["term_accuracy_strict"]), pct(g["term_accuracy_lenient"])))
table(["forhold", "klip", "ord", "WER %", "CER %", "fagord, præcis form %", "fagord, enhver bøjning %"],
      condition_rows, right={1, 2, 3, 4, 5, 6}, empty="tom")
on_fleurs = not metrics.counts_terms(dataset)  # "fleurs-..." datasets
if not condition_rows:
    print("Tom på FLEURS: tabellen fyldes, når du har kørt wt-train eval på dine egne optagelser." if on_fleurs
          else "Ingen af klippene har et optageforhold.")

## Pr. fagord

In [ ]:
# The English terms of spelling.yaml, counted only on the owner's own takes (metrics.counts_terms).
spelling = load_spelling(paths.SPELLING_YAML) if paths.SPELLING_YAML.is_file() else None
seen, strict, lenient = Counter(), Counter(), Counter()
if spelling is not None and metrics.counts_terms(dataset):
    for r in rows:
        ref = spelling.find_terms(r["reference"])
        hyp = spelling.find_terms(spelling.apply(r["hypothesis"]))
        ref_terms, hyp_terms = Counter(h.term for h in ref), Counter(h.term for h in hyp)
        seen.update(ref_terms)
        lenient.update(ref_terms & hyp_terms)  # the same term, any form
        same_form = Counter((h.term, h.form.casefold()) for h in ref) & Counter((h.term, h.form.casefold()) for h in hyp)
        for (term, _), k in same_form.items():
            strict[term] += k
term_rows = [(term, n, strict[term], lenient[term], da(strict[term] / n * 100, 1), da(lenient[term] / n * 100, 1))
             for term, n in sorted(seen.items(), key=lambda kv: (-(kv[1] - strict[kv[0]]), kv[0]))]
table(["fagord", "i referencerne", "præcis form", "enhver bøjning", "præcis form %", "enhver bøjning %"],
      term_rows, right={1, 2, 3, 4, 5}, empty="tom")
if not term_rows:
    print("Tom på FLEURS: tabellen fyldes, når du har kørt wt-train eval på dine egne optagelser." if on_fleurs
          else "Ingen fagord fra spelling.yaml i referencerne.")
else:
    print(f"i alt: {sum(strict.values())} af {sum(seen.values())} i præcis form "
          f"(rækkerne siger {sum(r['terms_strict'] for r in rows)} af {sum(r['terms'] for r in rows)})")